# U3 - Integrante 2 - Integración end-to-end, DataOps y hardening: Ballenas (`es_ballena`)

**Proyecto Sello · BTC/USDT (Binance)** — una sola ejecución que une lo construido en U1 (batch) y U2 (streaming), con compuertas, observabilidad y registro.

**Pregunta de mi dimensión:** ¿Se puede clasificar probabilísticamente una operación institucional/ballena (> 50 000 USDT) según la liquidez y la dirección del ejecutor?

## Qué hace este notebook (S13 integración y DataOps + S14 revisión técnica)

| Sección | Qué comprueba o produce | Criterio de la rúbrica U3 |
|---|---|---|
| 1 | 9 compuertas previas y una compuerta **demostrada fallando** | DataOps (15 %) |
| 2 | Batch sobre lo capturado en vivo: calidad, percentiles p50/p95/p99, compuerta de plausibilidad | Integración (20 %) |
| 3 | Indicador **U1 (offline)** frente a indicador **U2 (en vivo)** con el mismo modelo de mi dimensión | Integración (20 %) |
| 4 | Streaming en ventana fija: modelo U1 por operación + serie de tiempo por barra, error en producción | Integración (20 %) |
| 5 | Publicación en Pushgateway y **confirmación en Prometheus** comparando el valor recién publicado | DataOps (15 %) |
| 6 | Revisión de hardening del `compose.yml` real | Hardening (10 %) |
| 7 | Matriz de trazabilidad de mi fila, manifiesto, **comparación de dos corridas** y bitácora | Documentación (15 %), Demo (15 %) |

**Cómo se conecta con U1 y U2:** lee el Silver, el modelo ganador y las métricas de U1 desde `artifacts/`, y los eventos persistidos, el modelo de serie de tiempo y las métricas de U2 desde `artifacts/integrante2/`. **Escribe solo en `artifacts/integrante2/u3/`** (no borra nada de U2).

> **Antes de ejecutar:** `docker compose up -d` (con `bridge-binance`, `kafka`, `prometheus`, `pushgateway`, `grafana`), el notebook de U1 y el de U2 del integrante 2 ya ejecutados, y una copia de `compose.yml` dentro de `notebooks/` (para la revisión de hardening). **Ejecútalo dos veces** para poder comparar los manifiestos.
> Tiempo: ≈ 3–5 min por corrida. Variables: `DURACION_S`, `UMBRAL_LAT_MS`, `COMPOSE_PATH`, `STRICT_PROMETHEUS`.

## 0. Sesión Spark, parámetros y funciones

In [1]:
import os, json, time, glob, shutil, threading, uuid, urllib.request, urllib.parse
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pyspark

from pyspark.sql import SparkSession, functions as F, Window
from pyspark.sql.types import StructType, StructField, LongType, DoubleType, BooleanType, StringType
from pyspark.ml import Pipeline, PipelineModel
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.functions import vector_to_array
from pyspark.ml.regression import LinearRegression, RandomForestRegressor
from pyspark.ml.evaluation import RegressionEvaluator

# ---------------- Identidad y dimension ----------------
INTEGRANTE  = 2
DIMENSION   = "ballenas"            # etiqueta usada en las metricas de Prometheus
TARGET_COL  = "n_ballenas"         # variable de la serie de tiempo (S10)

# ---------------- Rutas (U1 en la raiz de artifacts; U2 en su carpeta) ----------------
WORK      = os.getenv("WORK_DIR", "/home/jovyan/work")
ARTIFACTS = os.getenv("ARTIFACTS_DIR", f"{WORK}/artifacts")      # salidas de U1 (silver, modelos ganadores)
OUT       = f"{ARTIFACTS}/integrante{INTEGRANTE}"                  # salidas de ESTE notebook
CKPT      = f"{OUT}/u3/checkpoints"
RUN_ID    = datetime.now().strftime("%m%d%H%M%S")
os.makedirs(OUT, exist_ok=True)
os.makedirs(CKPT, exist_ok=True)

# ---------------- Infraestructura (nombres del compose.yml) ----------------
KAFKA_BOOTSTRAP = os.getenv("KAFKA_BOOTSTRAP_SERVERS", "kafka:9092")
TOPIC           = os.getenv("KAFKA_TOPIC", "binance-btc-trades")
PUSHGATEWAY     = os.getenv("PUSHGATEWAY_URL", "http://pushgateway:9091")
PROMETHEUS      = os.getenv("PROMETHEUS_URL", "http://prometheus:9090")
STREAM_SOURCE   = os.getenv("STREAM_SOURCE", "kafka")             # "files": solo para pruebas sin Kafka
START_OFFSETS   = os.getenv("STARTING_OFFSETS", "latest")
TEST_DIR        = f"{OUT}/_test_stream"

# ---------------- Parametros del pipeline ----------------
BAR_SECONDS    = int(os.getenv("BAR_SECONDS", "10"))
BAR_WIDTH      = f"{BAR_SECONDS} seconds"
WATERMARK      = os.getenv("WATERMARK", "5 seconds")
LIVE_SECONDS   = int(os.getenv("LIVE_SECONDS", "150"))
WAIT_SCALE     = float(os.getenv("WAIT_SCALE", "1.0"))            # < 1.0 acorta las esperas (solo pruebas)
UMBRAL_BALLENA = 50000.0
RANGO_PRECIO   = (10000.0, 300000.0)                              # mismos limites que consumer_crypto.py
RANGO_QTY      = (0.000001, 500.0)
SEED           = 42
FEATS = ["n_trades", "volumen_btc", "volumen_usdt", "qty_media", "qty_max", "n_ballenas",
         "vol_ballenas_usdt", "pct_buyer_maker", "rango_rel", "retorno"]   # sin nivel de precio (evita deriva)

RESUMEN = {"integrante": INTEGRANTE, "dimension": DIMENSION, "target_serie": TARGET_COL,
           "bar_seconds": BAR_SECONDS, "watermark": WATERMARK, "run_id": RUN_ID}

builder = (SparkSession.builder
           .appName(f"U3-Integrante{INTEGRANTE}-{DIMENSION}")
           .master("local[*]")
           .config("spark.ui.port", "4040")
           .config("spark.sql.shuffle.partitions", "4")
           .config("spark.driver.memory", os.getenv("SPARK_DRIVER_MEMORY", "4g"))
           .config("spark.sql.session.timeZone", "UTC")
           .config("spark.sql.streaming.forceDeleteTempCheckpointLocation", "true"))
if STREAM_SOURCE == "kafka":   # conector Kafka alineado con la version de pyspark instalada
    builder = builder.config("spark.jars.packages", f"org.apache.spark:spark-sql-kafka-0-10_2.12:{pyspark.__version__}")
spark = builder.getOrCreate()
spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version} | fuente: {STREAM_SOURCE} | topic: {TOPIC} | salidas U2: {OUT}")

Spark 3.5.0 | fuente: kafka | topic: binance-btc-trades | salidas U2: /home/jovyan/work/artifacts/integrante2


In [2]:
TEST_TOPIC     = f"u2-test-i{INTEGRANTE}-{RUN_ID}"       # tardio / watermark
TEST_TOPIC_DUP = f"u2-testdup-i{INTEGRANTE}-{RUN_ID}"    # duplicados
TEST_TOPIC_VAL = f"u2-testval-i{INTEGRANTE}-{RUN_ID}"    # validacion esquema / rango

def esperar(seg, msg=None):
    s = seg * WAIT_SCALE
    if msg:
        print(f"... {msg} ({s:.0f}s)")
    time.sleep(s)

def ultimo(q):
    return q.lastProgress or {}

# ---------------- Publicacion / lectura (Kafka o archivos) ----------------
_producer = None
def publicar(topic, evento, key="BTCUSDT"):
    """Publica un evento JSON (str = mensaje crudo, util para probar mensajes malformados)."""
    global _producer
    valor = evento if isinstance(evento, str) else json.dumps(evento)
    if STREAM_SOURCE == "kafka":
        from kafka import KafkaProducer
        if _producer is None:
            _producer = KafkaProducer(bootstrap_servers=KAFKA_BOOTSTRAP)
        _producer.send(topic, key=key.encode(), value=valor.encode()).get(timeout=15)
        _producer.flush()
    else:
        d = f"{TEST_DIR}/{topic}"; os.makedirs(d, exist_ok=True)
        tmp = f"{d}/.tmp_{uuid.uuid4().hex}"
        with open(tmp, "w") as f:
            f.write(valor + "\n")
        os.rename(tmp, f"{d}/ev_{time.time_ns()}.json")

def evento_test(trade_id, ts_ms, price=93000.0, qty=0.01, maker=False):
    return {"tipoEvento": "crypto.trade", "tradeId": trade_id, "symbol": "BTCUSDT", "price": price,
            "quantity": qty, "quoteQuantity": round(price * qty, 8), "isBuyerMaker": maker,
            "origen": f"test-integrante{INTEGRANTE}", "timestamp": ts_ms}

def consumir_test(topic, timeout_s=8):
    """Consume (grupo propio, desde el inicio) los mensajes de un topic de prueba."""
    out = []
    if STREAM_SOURCE == "kafka":
        from kafka import KafkaConsumer
        c = KafkaConsumer(topic, bootstrap_servers=KAFKA_BOOTSTRAP, group_id=f"u2-i{INTEGRANTE}-check-{uuid.uuid4().hex[:6]}",
                          auto_offset_reset="earliest", consumer_timeout_ms=int(timeout_s * 1000))
        for m in c:
            out.append((m.partition, m.offset, m.value.decode("utf-8", errors="replace")))
        c.close()
    else:
        for i, p in enumerate(sorted(glob.glob(f"{TEST_DIR}/{topic}/ev_*"))):
            out.append((0, i, open(p).read().strip()))
    return out

def leer_stream(topic=None, offsets=None):
    """Stream crudo: key, value, topic, partition, offset, kafka_ts."""
    topic = topic or TOPIC
    if STREAM_SOURCE == "kafka":
        return (spark.readStream.format("kafka")
                .option("kafka.bootstrap.servers", KAFKA_BOOTSTRAP)
                .option("subscribe", topic)
                .option("startingOffsets", offsets or START_OFFSETS)
                .option("failOnDataLoss", "false")
                .load()
                .select(F.col("key").cast("string").alias("key"), F.col("value").cast("string").alias("value"),
                        "topic", "partition", "offset", F.col("timestamp").alias("kafka_ts")))
    d = f"{TEST_DIR}/{topic}"; os.makedirs(d, exist_ok=True)
    return (spark.readStream.format("text").load(d)
            .select(F.lit(None).cast("string").alias("key"), "value", F.lit(topic).alias("topic"),
                    F.lit(0).alias("partition"), F.lit(0).cast("long").alias("offset"),
                    F.current_timestamp().alias("kafka_ts")))

def tasa_topic(topic, seg=5):
    """(particiones, offset_total, eventos_por_segundo) del topic."""
    if STREAM_SOURCE == "kafka":
        from kafka import KafkaConsumer, TopicPartition
        c = KafkaConsumer(bootstrap_servers=KAFKA_BOOTSTRAP)
        parts = sorted(c.partitions_for_topic(topic) or [])
        tps = [TopicPartition(topic, p) for p in parts]
        f1 = sum(c.end_offsets(tps).values()); time.sleep(seg)
        f2 = sum(c.end_offsets(tps).values()); c.close()
        return len(parts), f2, (f2 - f1) / seg
    n = lambda: sum(len(open(p).read().splitlines()) for p in glob.glob(f"{TEST_DIR}/{topic}/ev_*"))
    n1 = n(); time.sleep(seg); n2 = n()
    return 1, n2, (n2 - n1) / seg

# ---------------- Contrato del evento (producido por bridge_binance_kafka.py) ----------------
schema_evento = StructType([
    StructField("tipoEvento", StringType()), StructField("tradeId", LongType()),
    StructField("symbol", StringType()),     StructField("price", DoubleType()),
    StructField("quantity", DoubleType()),   StructField("quoteQuantity", DoubleType()),
    StructField("isBuyerMaker", BooleanType()), StructField("origen", StringType()),
    StructField("timestamp", LongType()),    # = campo T de Binance, en MILISEGUNDOS (tiempo del evento)
])

def parsear(df_raw):
    """JSON -> columnas tipadas + campos de observabilidad (S9): isValid, processedAt, latencyMs."""
    e = df_raw.select(F.from_json("value", schema_evento).alias("e"), "topic", "partition", "offset", "kafka_ts").select("e.*", "topic", "partition", "offset", "kafka_ts")
    e = (e.withColumnRenamed("tradeId", "trade_id").withColumnRenamed("timestamp", "event_ts_ms")
          .withColumnRenamed("quantity", "qty").withColumnRenamed("quoteQuantity", "quote_qty")
          .withColumnRenamed("isBuyerMaker", "is_buyer_maker"))
    return (e
        .withColumn("event_time", (F.col("event_ts_ms") / 1000).cast("timestamp"))
        .withColumn("is_buyer_maker_num", F.when(F.col("is_buyer_maker") == True, 1.0).otherwise(0.0))
        .withColumn("es_ballena", F.when(F.col("quote_qty") > UMBRAL_BALLENA, 1.0).otherwise(0.0))
        .withColumn("processedAt", F.current_timestamp())
        .withColumn("latencyMs", ((F.col("processedAt").cast("double") * 1000) - F.col("event_ts_ms")).cast("long"))
        .withColumn("isValid", F.coalesce(
            F.col("trade_id").isNotNull() & F.col("event_ts_ms").isNotNull() & F.col("is_buyer_maker").isNotNull()
            & (F.col("tipoEvento") == "crypto.trade")
            & F.col("price").between(*RANGO_PRECIO) & F.col("qty").between(*RANGO_QTY)
            & F.col("quote_qty").isNotNull(), F.lit(False))))

def ev_stream(topic=None, offsets=None):
    return parsear(leer_stream(topic, offsets))

def validar_evento(txt):
    """Misma logica que consumer_crypto.py: 1) esquema, 2) rango plausible."""
    try:
        p = json.loads(txt)
    except Exception:
        return "INVALIDO_ESQUEMA", "JSON malformado"
    req = ["tipoEvento", "tradeId", "symbol", "price", "quantity", "quoteQuantity", "timestamp"]
    if not isinstance(p, dict) or any(p.get(c) is None for c in req) or p.get("tipoEvento") != "crypto.trade":
        return "INVALIDO_ESQUEMA", "faltan campos o tipoEvento incorrecto"
    if not (RANGO_PRECIO[0] <= p["price"] <= RANGO_PRECIO[1]):
        return "ALERTA_RANGO", f"precio fuera de rango plausible: {p['price']}"
    if not (RANGO_QTY[0] <= p["quantity"] <= RANGO_QTY[1]):
        return "ALERTA_RANGO", f"cantidad fuera de rango plausible: {p['quantity']}"
    return "OK", "evento valido"

# ---------------- Barras de tiempo: MISMA funcion para historico (batch) y vivo (streaming) ----------------
def agregar_barras(df, tcol="event_time", ancho=BAR_WIDTH):
    orden = F.struct("event_ts_ms", "trade_id")
    g = df.groupBy(F.window(tcol, ancho).alias("w")).agg(
        F.count("*").alias("n_trades"),
        F.sum("qty").alias("volumen_btc"),
        F.sum("quote_qty").alias("volumen_usdt"),
        F.avg("qty").alias("qty_media"),
        F.max("qty").alias("qty_max"),
        F.sum("es_ballena").alias("n_ballenas"),
        F.sum(F.when(F.col("es_ballena") == 1.0, F.col("quote_qty")).otherwise(0.0)).alias("vol_ballenas_usdt"),
        F.avg("is_buyer_maker_num").alias("pct_buyer_maker"),
        F.min("price").alias("precio_min"), F.max("price").alias("precio_max"),
        F.min_by("price", orden).alias("precio_apertura"), F.max_by("price", orden).alias("precio_cierre"))
    return g.select(
        F.col("w.start").alias("bar_start"), F.col("w.end").alias("bar_end"),
        "n_trades", "volumen_btc", "volumen_usdt", "qty_media", "qty_max", "n_ballenas",
        "vol_ballenas_usdt", "pct_buyer_maker",
        ((F.col("precio_max") - F.col("precio_min")) / (F.col("volumen_usdt") / F.col("volumen_btc"))).alias("rango_rel"),
        ((F.col("precio_cierre") - F.col("precio_apertura")) / F.col("precio_apertura")).alias("retorno"))

# ---------------- Capturador de microlotes (foreachBatch) para las demostraciones ----------------
CAPT = {}
def capturador(nombre, claves):
    CAPT[nombre] = {"lotes": [], "filas": {}}
    def f(df, bid):
        pdf = df.toPandas()
        CAPT[nombre]["lotes"].append((int(bid), len(pdf)))
        for r in pdf.to_dict("records"):
            CAPT[nombre]["filas"][tuple(r[k] for k in claves)] = r
    return f

def inspeccionar_checkpoint(ruta):
    print(f"Checkpoint: {ruta}")
    for sub in sorted(os.listdir(ruta)):
        n = sum(len(fs) for _, _, fs in os.walk(os.path.join(ruta, sub)))
        print(f"  {sub}/  ({n} archivos)")
    arch = [a for a in glob.glob(f"{ruta}/offsets/*") if os.path.basename(a).isdigit()]
    if arch:
        ult = sorted(arch, key=lambda x: int(os.path.basename(x)))[-1]
        print(f"  ultimo offsets/{os.path.basename(ult)}:")
        print("   ", open(ult).read().replace("\n", "\n    ")[:600])

# ---------------- Pushgateway y Prometheus (stdlib, sin dependencias extra) ----------------
def push(job, instance, metricas, labels=None, metodo="PUT"):
    lab = ",".join(f'{k}="{v}"' for k, v in (labels or {}).items())
    lineas = []
    for nombre, valor in metricas.items():
        if valor is None:
            continue
        v = float(valor)
        if np.isnan(v) or np.isinf(v):
            continue
        lineas.append(f"# TYPE {nombre} gauge")
        lineas.append(f"{nombre}{{{lab}}} {v}" if lab else f"{nombre} {v}")
    req = urllib.request.Request(f"{PUSHGATEWAY}/metrics/job/{job}/instance/{instance}",
                                 data=("\n".join(lineas) + "\n").encode(), method=metodo,
                                 headers={"Content-Type": "text/plain; version=0.0.4"})
    try:
        urllib.request.urlopen(req, timeout=5).read()
        return True
    except Exception as ex:
        print(f"[PUSH] no se pudo enviar a {PUSHGATEWAY}: {ex}")
        return False

def promql(q):
    url = f"{PROMETHEUS}/api/v1/query?" + urllib.parse.urlencode({"query": q})
    try:
        return json.load(urllib.request.urlopen(url, timeout=5))["data"]["result"]
    except Exception as ex:
        print(f"[PROMQL] Prometheus no disponible ({ex})")
        return None

def mostrar_promql(titulo, q):
    r = promql(q)
    print(f"{titulo}\n  PromQL: {q}")
    if r is None:
        return None
    if not r:
        print("  (sin series todavia)")
    for s in r:
        print("  ", {k: v for k, v in s["metric"].items() if k in ("job", "instance", "topic", "consumergroup", "dimension")}, "=", s["value"][1])
    return r
print("Funciones listas.")

Funciones listas.


In [3]:
# ---------------- Rutas de lo que ya construyeron U1 y U2 ----------------
OUT3             = f"{OUT}/u3"                                              # salidas de ESTE notebook
RUTA_SILVER      = f"{ARTIFACTS}/silver_integrante{INTEGRANTE}"             # U1
RUTA_MODELO_U1   = f"{ARTIFACTS}/modelo_ganador_integrante{INTEGRANTE}"     # U1
RUTA_METRICAS_U1 = f"{ARTIFACTS}/metricas_integrante{INTEGRANTE}.json"      # U1
RUTA_EV          = f"{OUT}/stream/eventos_obs"                              # U2: eventos de streaming persistidos
RUTA_MODELO_TS   = f"{OUT}/modelo_series_tiempo"                            # U2: serie de tiempo
GRAFANA          = os.getenv("GRAFANA_URL", "http://grafana:3000")
COMPOSE_PATH     = os.getenv("COMPOSE_PATH", f"{WORK}/compose.yml")         # copiar compose.yml a notebooks/
os.makedirs(OUT3, exist_ok=True)

# ---------------- Parametros de la ejecucion integrada ----------------
LBL            = "es_ballena"      # columna real que predice el modelo U1 de mi dimension
DURACION_S     = int(os.getenv("DURACION_S", "90"))        # ventana fija de streaming
MIN_FILAS      = int(os.getenv("MIN_FILAS", "500"))        # minimo de eventos persistidos para confiar en el batch
UMBRAL_LAT_MS  = int(os.getenv("UMBRAL_LAT_MS", "30000"))  # plausibilidad: p50 de latencia por debajo de 30 s
UMBRAL_P95_ADV = int(os.getenv("UMBRAL_P95_ADV", "60000")) # advertencia: p95 por encima de 60 s
STRICT_PROM    = os.getenv("STRICT_PROMETHEUS", "1") == "1"

import platform
INICIO = time.time()
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
manifiesto = {"runId": RUN_ID, "integrante": INTEGRANTE, "dimension": DIMENSION,
              "sparkVersion": spark.version, "pyspark": pyspark.__version__, "python": platform.python_version(),
              "fuente": STREAM_SOURCE, "topic": TOPIC, "barSeconds": BAR_SECONDS, "watermark": WATERMARK,
              "umbrales": {"p50MaxMs": UMBRAL_LAT_MS, "p95AdvertenciaMs": UMBRAL_P95_ADV},
              "compuertas": {}, "demostracionFalla": {}, "batch": {}, "streaming": {}, "observabilidad": {},
              "hardening": {}, "advertencias": []}
print(f"Ejecucion integrada {RUN_ID} | integrante {INTEGRANTE} ({DIMENSION}) | salidas en {OUT3}")

Ejecucion integrada 20261113T195359Z | integrante 2 (ballenas) | salidas en /home/jovyan/work/artifacts/integrante2/u3


In [4]:
import platform
import re
import yaml
from pyspark.ml.evaluation import BinaryClassificationEvaluator

def http_ok(url):
    try:
        with urllib.request.urlopen(url, timeout=5) as r:
            return r.status == 200
    except Exception:
        return False

def kafka_filas(topic):
    if STREAM_SOURCE == "kafka":
        return (spark.read.format("kafka").option("kafka.bootstrap.servers", KAFKA_BOOTSTRAP).option("subscribe", topic)
                .option("startingOffsets", "earliest").option("endingOffsets", "latest").load().count())
    return sum(len(open(p).read().splitlines()) for p in glob.glob(f"{TEST_DIR}/{topic}/ev_*"))

def gate(fn):
    try:
        return bool(fn())
    except Exception:
        return False

def definir_compuertas():
    return {
        "Silver de U1 existe": lambda: os.path.isdir(RUTA_SILVER),
        "Modelo ganador de U1 existe": lambda: os.path.isdir(RUTA_MODELO_U1),
        "Metricas offline de U1 (JSON) existen": lambda: os.path.isfile(RUTA_METRICAS_U1),
        "Eventos de streaming de U2 persistidos (Parquet)": lambda: os.path.isdir(RUTA_EV),
        "Modelo de serie de tiempo de U2 existe": lambda: os.path.isdir(RUTA_MODELO_TS),
        f"Kafka: topic {TOPIC} con datos": lambda: kafka_filas(TOPIC) > 0,
        "Pushgateway listo": lambda: http_ok(f"{PUSHGATEWAY}/-/ready"),
        "Prometheus listo": lambda: http_ok(f"{PROMETHEUS}/-/ready"),
        "Grafana saludable": lambda: http_ok(f"{GRAFANA}/api/health"),
    }

def correr_compuertas(spec):
    res = {n: gate(f) for n, f in spec.items()}
    for n, ok in res.items():
        print(("OK    " if ok else "FALLA ") + n)
    return res

def exigir(res):
    fallidas = [n for n, ok in res.items() if not ok]
    assert not fallidas, "Compuertas que fallan: " + str(fallidas)

def num(v):
    """Valor JSON-seguro."""
    if v is None:
        return None
    try:
        v = float(v)
        return None if (np.isnan(v) or np.isinf(v)) else v
    except Exception:
        return None

def evaluar_u1_df(pred):
    """Indicador de mi dimension sobre un DataFrame de Spark con 'label' y 'prediction' (clasificacion)."""
    c = {(r["label"], r["prediction"]): r["count"] for r in pred.groupBy("label", "prediction").count().collect()}
    tp, fp, fn, tn = c.get((1.0, 1.0), 0), c.get((0.0, 1.0), 0), c.get((1.0, 0.0), 0), c.get((0.0, 0.0), 0)
    n, pos = tp + fp + fn + tn, tp + fn
    m = {"n": int(n), "positivos": int(pos), "accuracy": (tp + tn) / n, "baseline_acc": max(pos, n - pos) / n,
         "precision_pos": tp / (tp + fp) if (tp + fp) else None, "recall_pos": tp / pos if pos else None,
         "tp": int(tp), "fp": int(fp), "fn": int(fn), "tn": int(tn), "auc_roc": None, "auc_pr": None}
    if 0 < pos < n:
        for k, nombre in (("auc_roc", "areaUnderROC"), ("auc_pr", "areaUnderPR")):
            m[k] = float(BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="rawPrediction", metricName=nombre).evaluate(pred))
    return m

def evaluar_ventana(pdf):
    """Lo mismo, sobre las filas (pandas) que proceso el stream durante la ventana."""
    y, p = pdf[LBL].astype(float), pdf["prediction"].astype(float)
    tp, fp, fn = int(((y == 1) & (p == 1)).sum()), int(((y == 0) & (p == 1)).sum()), int(((y == 1) & (p == 0)).sum())
    pos = int((y == 1).sum())
    return {"n": int(len(pdf)), "positivos": pos, "accuracy": float((y == p).mean()),
            "baseline_acc": float(max(y.mean(), 1 - y.mean())),
            "precision_pos": tp / (tp + fp) if (tp + fp) else None, "recall_pos": tp / pos if pos else None,
            "tp": tp, "fp": fp, "fn": fn}
print("Funciones de la ejecucion integrada listas.")

Funciones de la ejecucion integrada listas.


## 1. Compuertas previas (*preflight*)

Nueve dependencias se comprueban **antes** de que Spark lea nada. Si una falla, la ejecución se detiene aquí con el nombre exacto. Recuerda que `artifacts/` no se versiona: una copia clonada del repositorio no trae los artefactos de U1 ni de U2, y esta compuerta lo dice con nombre.

In [5]:
# Compuertas PREVIAS: si una dependencia falta, la ejecucion se detiene aqui con el nombre exacto, no minutos despues dentro de Spark.
res = correr_compuertas(definir_compuertas())
manifiesto["compuertas"] = res
exigir(res)

OK    Silver de U1 existe
OK    Modelo ganador de U1 existe
OK    Metricas offline de U1 (JSON) existen
OK    Eventos de streaming de U2 persistidos (Parquet)
OK    Modelo de serie de tiempo de U2 existe
OK    Kafka: topic binance-btc-trades con datos
OK    Pushgateway listo
OK    Prometheus listo
OK    Grafana saludable


**Demostración de una compuerta que falla.** Se apunta a un puerto equivocado a propósito para mostrar que la ejecución se detiene y deja constancia.

In [6]:
# Demostracion obligatoria: una compuerta que FALLA detiene la ejecucion. Se apunta a un puerto equivocado a proposito
# y se captura el AssertionError para poder seguir con el notebook y dejar la evidencia en el manifiesto.
spec_demo = {"Pushgateway listo (puerto equivocado a proposito)": lambda: http_ok(PUSHGATEWAY.rsplit(":", 1)[0] + ":1/-/ready")}
try:
    exigir(correr_compuertas(spec_demo))
    manifiesto["demostracionFalla"] = {"detuvo": False}
    print("ERROR: la compuerta deberia haber fallado")
except AssertionError as e:
    print("\nLa ejecucion SE DETUVO como debia ->", e)
    manifiesto["demostracionFalla"] = {"detuvo": True, "mensaje": str(e)}

FALLA Pushgateway listo (puerto equivocado a proposito)

La ejecucion SE DETUVO como debia -> Compuertas que fallan: ['Pushgateway listo (puerto equivocado a proposito)']


## 2. Batch sobre lo capturado en vivo: calidad y percentiles

Se lee el Parquet que U2 persistió (`stream/eventos_obs`). Las compuertas de calidad cubren existencia, completitud, rango y **plausibilidad**. La de plausibilidad usa la **mediana (p50)**: el promedio puede describir a ningún evento real, y el p95 queda como **advertencia**, no como falla, porque detener todo por una cola larga sería una decisión de negocio y no técnica. Umbral de p50: 30 s (la barra es de 10 s y el *watermark* de 5 s: una mediana mayor indica retraso acumulado).

In [7]:
ev_all = spark.read.parquet(RUTA_EV)
total = ev_all.count()
validos = ev_all.filter("isValid")
nulos = ev_all.agg(F.sum(F.col("isValid").isNull().cast("int")).alias("isValid")).first()["isValid"] or 0
nulos_lat = validos.agg(F.sum(F.col("latencyMs").isNull().cast("int")).alias("n")).first()["n"] or 0

kpi = validos.agg(
    F.avg("latencyMs").alias("promedio"),
    F.expr("percentile_approx(latencyMs, 0.5)").alias("p50"),
    F.expr("percentile_approx(latencyMs, 0.95)").alias("p95"),
    F.expr("percentile_approx(latencyMs, 0.99)").alias("p99"),
    F.max("latencyMs").alias("max"),
    F.avg((F.col("latencyMs") > UMBRAL_LAT_MS).cast("int")).alias("fraccion_mayor_umbral")).first()
ratio_validos = ev_all.agg(F.avg(F.col("isValid").cast("int"))).first()[0]

calidad = {
    f"Hay filas (>= {MIN_FILAS})": total >= MIN_FILAS,
    "isValid sin nulos": nulos == 0,
    "latencyMs sin nulos (eventos validos)": nulos_lat == 0,
    "ratioValidos en [0, 1]": 0.0 <= ratio_validos <= 1.0,
    f"Latencia tipica (p50) menor a {UMBRAL_LAT_MS // 1000} s": kpi["p50"] < UMBRAL_LAT_MS,
}
for n, ok in calidad.items():
    print(("OK    " if ok else "FALLA ") + n)
exigir(calidad)

print(f"\nEventos persistidos: {total:,} | ratio de validos: {ratio_validos:.4f}")
print(f"Latencia (ms): promedio {kpi['promedio']:.0f} | p50 {kpi['p50']} | p95 {kpi['p95']} | p99 {kpi['p99']} | max {kpi['max']}")
print(f"Eventos con latencia > {UMBRAL_LAT_MS // 1000} s: {kpi['fraccion_mayor_umbral']:.1%}")
adv = []
if kpi["p95"] > UMBRAL_P95_ADV:
    adv.append(f"p95 de latencia ({kpi['p95']} ms) mayor a {UMBRAL_P95_ADV // 1000} s: la cola es mala aunque la mediana sea buena")
if kpi["promedio"] > 2 * kpi["p50"]:
    adv.append(f"el promedio ({kpi['promedio']:.0f} ms) supera 2x la mediana ({kpi['p50']} ms): no representa al evento tipico")
if ratio_validos < 0.99:
    adv.append(f"ratio de validos {ratio_validos:.4f} menor a 0.99: llegan eventos fuera de esquema o de rango")
for a in adv:
    print("ADVERTENCIA:", a)
manifiesto["advertencias"] = adv
manifiesto["batch"] = {"filas": int(total), "ratioValidos": num(ratio_validos), "latenciaPromedioMs": num(kpi["promedio"]),
                       "latenciaP50Ms": num(kpi["p50"]), "latenciaP95Ms": num(kpi["p95"]), "latenciaP99Ms": num(kpi["p99"]),
                       "latenciaMaxMs": num(kpi["max"]), "fraccionMayorUmbral": num(kpi["fraccion_mayor_umbral"]), "calidad": calidad}

OK    Hay filas (>= 500)
OK    isValid sin nulos
OK    latencyMs sin nulos (eventos validos)
OK    ratioValidos en [0, 1]
OK    Latencia tipica (p50) menor a 30 s

Eventos persistidos: 743 | ratio de validos: 1.0000
Latencia (ms): promedio 3129 | p50 2673 | p95 5025 | p99 10759 | max 14170
Eventos con latencia > 30 s: 0.0%


## 3. Indicador U1 (offline) frente a indicador U2 (en vivo): Recall de ballenas

El **mismo modelo ganador de U1** se aplica sobre los eventos capturados del WebSocket. La diferencia entre ambos números es lo que cambió entre el CSV histórico y el flujo real.

In [8]:
# Indicador U1 (offline, del informe de U1) frente al indicador U2 (el MISMO modelo U1 aplicado sobre los eventos capturados en vivo)
modelo_u1 = PipelineModel.load(RUTA_MODELO_U1)
met_u1 = json.load(open(RUTA_METRICAS_U1))
ganador_u1 = met_u1["ganador"]
res_u1 = met_u1["resultados"][ganador_u1]

pred_cap = modelo_u1.transform(validos.withColumn("label", F.col(LBL).cast("double")))
res_u2 = evaluar_u1_df(pred_cap)

IND_KEY = "recall_pos"
ind_u1, ind_u2 = num(res_u1.get(IND_KEY)), num(res_u2.get(IND_KEY))
print(f"Modelo ganador de U1: {ganador_u1}")
print(f"Indicador de mi dimension: {IND_KEY} | U1 (offline, CSV 2026-01-05) = {ind_u1} | U2 (eventos capturados en vivo) = {ind_u2}")
print({k: (round(v, 6) if isinstance(v, float) else v) for k, v in res_u2.items() if not isinstance(v, dict)})
if ind_u2 is None:
    manifiesto["advertencias"].append(f"indicador {IND_KEY} no calculable sobre los eventos capturados (clase positiva ausente)")
manifiesto["batch"]["modeloU1"] = ganador_u1
manifiesto["batch"]["indicadorKey"] = IND_KEY
manifiesto["batch"]["indicadorU1"] = ind_u1
manifiesto["batch"]["indicadorU2"] = ind_u2
manifiesto["batch"]["evaluacionCapturados"] = {k: v for k, v in res_u2.items()}

Modelo ganador de U1: RandomForest_15_ponderado
Indicador de mi dimension: recall_pos | U1 (offline, CSV 2026-01-05) = 0.9939516129032258 | U2 (eventos capturados en vivo) = None
{'n': 743, 'positivos': 0, 'accuracy': 1.0, 'baseline_acc': 1.0, 'precision_pos': None, 'recall_pos': None, 'tp': 0, 'fp': 0, 'fn': 0, 'tn': 743, 'auc_roc': None, 'auc_pr': None}


## 4. Streaming en ventana fija: modelo U1 y serie de tiempo

Dos consultas sobre el mismo flujo deduplicado y con *watermark*: (A) el modelo de U1 sobre **cada operación**; (B) el modelo de serie de tiempo (n_ballenas de la barra siguiente) sobre **cada barra cerrada**. A diferencia de U2, la consulta **termina**: una ejecución integrada tiene que acabar para poder comprobarse y registrarse.

In [9]:
modelo_ts = PipelineModel.load(RUTA_MODELO_TS)
COLS_V = ["trade_id", "event_ts_ms", "price", "qty", "quote_qty", "is_buyer_maker_num", "es_ballena"]
filas_u1, lotes_u1, barras_ts = [], [], []

def procesar_trades(df, bid):                       # (A) modelo U1 sobre cada operacion
    pdf = modelo_u1.transform(df).select(*COLS_V, "prediction").toPandas()
    lotes_u1.append({"batchId": int(bid), "filas": int(len(pdf))})
    filas_u1.extend(pdf.to_dict("records"))

def procesar_barras(df, bid):                       # (B) modelo de serie de tiempo sobre cada barra cerrada
    if df.isEmpty():
        return
    cols = list(dict.fromkeys(["bar_start", TARGET_COL, "prediction"]))
    barras_ts.extend(modelo_ts.transform(df).select(*cols).toPandas().to_dict("records"))

def flujo():
    return ev_stream().filter("isValid").withWatermark("event_time", WATERMARK).dropDuplicates(["trade_id", "event_time"])

q_t = (flujo().writeStream.outputMode("append").queryName("u3_trades").foreachBatch(procesar_trades)
       .option("checkpointLocation", f"{CKPT}/{RUN_ID}/trades").trigger(processingTime="5 seconds").start())
q_b = (agregar_barras(flujo()).writeStream.outputMode("append").queryName("u3_barras").foreachBatch(procesar_barras)
       .option("checkpointLocation", f"{CKPT}/{RUN_ID}/barras").trigger(processingTime="5 seconds").start())

esperar(DURACION_S, f"ventana fija de streaming de {DURACION_S} s")
prog = {"trades": ultimo(q_t), "barras": ultimo(q_b)}
q_t.stop(); q_b.stop()

print("Micro-lotes con datos (modelo U1):", len([m for m in lotes_u1 if m["filas"] > 0]))
print("Operaciones procesadas en la ventana:", len(filas_u1), "| barras cerradas:", len(barras_ts))

... ventana fija de streaming de 90 s (90s)
Micro-lotes con datos (modelo U1): 18
Operaciones procesadas en la ventana: 1497 | barras cerradas: 7


### Error en producción
El error de mi dimensión sobre lo que acaba de procesar el stream, y el de la serie de tiempo frente a la **persistencia** (la barra siguiente igual a la actual). Dos corridas darán valores distintos: la fuente es en vivo, y eso es lo esperable.

In [10]:
assert filas_u1, "El stream no entrego ninguna fila: esta corriendo el puente (bridge-binance)?"

# (A) Error de mi dimension en produccion (sobre lo que acaba de procesar el stream)
ventana = evaluar_ventana(pd.DataFrame(filas_u1))
print("Dimension, ventana en vivo:", {k: (round(v, 6) if isinstance(v, float) else v) for k, v in ventana.items()})

# (B) Error de la serie de tiempo: prediccion de la barra t contra el valor real de la barra t+1, y contra la persistencia
bars = pd.DataFrame(barras_ts).sort_values("bar_start").reset_index(drop=True) if barras_ts else pd.DataFrame()
if len(bars):
    bars = bars.iloc[1:].reset_index(drop=True)      # la primera barra es parcial (el stream arranca a mitad de ventana)
err_m, err_p = [], []
for a, b in zip(bars.to_dict("records"), bars.to_dict("records")[1:]):
    if (b["bar_start"] - a["bar_start"]).total_seconds() == BAR_SECONDS:
        err_m.append(abs(float(a["prediction"]) - float(b[TARGET_COL])))
        err_p.append(abs(float(a[TARGET_COL]) - float(b[TARGET_COL])))
mae_m = float(np.mean(err_m)) if err_m else None
mae_p = float(np.mean(err_p)) if err_p else None
print(f"Serie de tiempo ({TARGET_COL}): pares comparables = {len(err_m)} | MAE modelo = {mae_m} | MAE persistencia = {mae_p}")
if not err_m:
    manifiesto["advertencias"].append("ventana demasiado corta: menos de dos barras cerradas, sin MAE de serie de tiempo")
manifiesto["streaming"] = {"duracionS": DURACION_S, "filas": len(filas_u1), "microlotes": lotes_u1, "barrasCerradas": len(barras_ts),
                           "paresComparables": len(err_m), "maeTs": num(mae_m), "maePersistencia": num(mae_p),
                           "ventana": {k: v for k, v in ventana.items()}}

Dimension, ventana en vivo: {'n': 1497, 'positivos': 0, 'accuracy': 0.997328, 'baseline_acc': 1.0, 'precision_pos': 0.0, 'recall_pos': None, 'tp': 0, 'fp': 4, 'fn': 0}
Serie de tiempo (n_ballenas): pares comparables = 5 | MAE modelo = 0.26563925318830733 | MAE persistencia = 0.0


## 5. Publicar en Pushgateway y confirmar en Prometheus

No basta con que cada métrica tenga *algún* valor: se compara con el que **acabo de publicar**. Si no, una corrida anterior podría darse por buena.

In [11]:
inst = f"integrante{INTEGRANTE}"
ok_c = sum(1 for v in manifiesto["compuertas"].values() if v)
metricas = {
    "cripto_int_batch_eventos_total": manifiesto["batch"]["filas"],
    "cripto_int_batch_validos_ratio": manifiesto["batch"]["ratioValidos"],
    "cripto_int_batch_latencia_p50_ms": manifiesto["batch"]["latenciaP50Ms"],
    "cripto_int_batch_latencia_p95_ms": manifiesto["batch"]["latenciaP95Ms"],
    "cripto_int_batch_latencia_p99_ms": manifiesto["batch"]["latenciaP99Ms"],
    "cripto_int_batch_fraccion_mayor_umbral": manifiesto["batch"]["fraccionMayorUmbral"],
    "cripto_int_indicador_u1": manifiesto["batch"]["indicadorU1"],
    "cripto_int_indicador_u2": manifiesto["batch"]["indicadorU2"],
    "cripto_int_stream_filas_total": manifiesto["streaming"]["filas"],
    "cripto_int_stream_barras_cerradas": manifiesto["streaming"]["barrasCerradas"],
    "cripto_int_ts_mae_modelo": manifiesto["streaming"]["maeTs"],
    "cripto_int_ts_mae_persistencia": manifiesto["streaming"]["maePersistencia"],
    "cripto_int_compuertas_ok": ok_c,
    "cripto_int_compuertas_total": len(manifiesto["compuertas"]),
    "cripto_int_ultima_ejecucion_timestamp": int(time.time()),
}
for k, v in manifiesto["streaming"]["ventana"].items():          # indicadores de la ventana en vivo
    if isinstance(v, (int, float)) and not isinstance(v, bool):
        metricas[f"cripto_int_ventana_{k}"] = v
metricas = {k: num(v) for k, v in metricas.items() if num(v) is not None}

ok_push = push("criptoanalytics_integracion", inst, metricas, labels={"dimension": DIMENSION})
print("Pushgateway:", "200 OK" if ok_push else "no respondio")

def consultar_prom(nombre):
    try:
        url = f"{PROMETHEUS}/api/v1/query?" + urllib.parse.urlencode({"query": f'{nombre}{{instance="{inst}"}}'})
        r = json.load(urllib.request.urlopen(url, timeout=5))["data"]["result"]
        return r[0]["value"][1] if r else None
    except Exception:
        return None

def coincide(obtenido, esperado):
    return obtenido is not None and abs(float(obtenido) - float(esperado)) <= 1e-6 * max(1.0, abs(float(esperado)))

confirmadas = {}
limite = time.time() + 90 * WAIT_SCALE
while time.time() < limite:
    confirmadas = {n: consultar_prom(n) for n in metricas}
    if all(coincide(confirmadas[n], v) for n, v in metricas.items()):
        break
    time.sleep(5 * WAIT_SCALE)
for n, v in metricas.items():
    print(f"{n}: {confirmadas[n]}  (publicado: {v})")
todas = all(coincide(confirmadas[n], v) for n, v in metricas.items())
manifiesto["observabilidad"] = {"publicadas": metricas, "confirmadasEnPrometheus": confirmadas, "todasEnPrometheus": bool(todas)}
if STRICT_PROM:
    assert todas, "Prometheus no devolvio los valores recien publicados"
elif not todas:
    print("AVISO: Prometheus no confirmo todos los valores (STRICT_PROMETHEUS=0).")

Pushgateway: 200 OK
cripto_int_batch_eventos_total: 743  (publicado: 743.0)
cripto_int_batch_validos_ratio: 1  (publicado: 1.0)
cripto_int_batch_latencia_p50_ms: 2673  (publicado: 2673.0)
cripto_int_batch_latencia_p95_ms: 5025  (publicado: 5025.0)
cripto_int_batch_latencia_p99_ms: 10759  (publicado: 10759.0)
cripto_int_batch_fraccion_mayor_umbral: 0  (publicado: 0.0)
cripto_int_indicador_u1: 0.9939516129032258  (publicado: 0.9939516129032258)
cripto_int_stream_filas_total: 1497  (publicado: 1497.0)
cripto_int_stream_barras_cerradas: 7  (publicado: 7.0)
cripto_int_ts_mae_modelo: 0.26563925318830733  (publicado: 0.26563925318830733)
cripto_int_ts_mae_persistencia: 0  (publicado: 0.0)
cripto_int_compuertas_ok: 9  (publicado: 9.0)
cripto_int_compuertas_total: 9  (publicado: 9.0)
cripto_int_ultima_ejecucion_timestamp: 1794599735  (publicado: 1794599735.0)
cripto_int_ventana_n: 1497  (publicado: 1497.0)
cripto_int_ventana_positivos: 0  (publicado: 0.0)
cripto_int_ventana_accuracy: 0.99732798

## 6. Hardening: revisión técnica de la configuración (S14)

Lista de comprobación sobre el `compose.yml` real: versión de imágenes, reinicio, `healthcheck`, `depends_on` con `service_healthy`, secretos fuera del archivo y puertos expuestos. Lo que falla se corrige en el `compose.yml` y se vuelve a ejecutar este notebook; lo que se **acepta** se registra como riesgo con su justificación.

In [12]:
# Revision de hardening (S14): lista de comprobacion sobre el compose.yml REAL del proyecto.
filas_h = []
if not os.path.isfile(COMPOSE_PATH):
    print(f"No encuentro {COMPOSE_PATH}. Copia compose.yml a la carpeta notebooks/ (se monta en {WORK}) o define COMPOSE_PATH.")
    manifiesto["hardening"] = {"disponible": False}
else:
    cfg = yaml.safe_load(open(COMPOSE_PATH, encoding="utf-8"))
    sv = cfg.get("services", {})
    def fila(punto, resultado, detalle):
        filas_h.append({"punto": punto, "resultado": resultado, "detalle": detalle})

    sin_ver = [n for n, s in sv.items() if "image" in s and (":" not in s["image"].split("/")[-1] or s["image"].endswith(":latest"))]
    fila("Version de las imagenes", "Cumple" if not sin_ver else "Falla", "todas con version exacta" if not sin_ver else f"sin version fija: {sin_ver}")
    sin_rst = [n for n, s in sv.items() if "restart" not in s]
    fila("Reinicio", "Cumple" if not sin_rst else "Falla", "restart definido en todos" if not sin_rst else f"sin restart: {sin_rst}")
    clave = [n for n in ("kafka", "prometheus", "pushgateway", "grafana") if n in sv]
    sin_hc = [n for n in clave if "healthcheck" not in sv[n]]
    fila("Salud (healthcheck)", "Cumple" if not sin_hc else "Falla", "servicios clave con healthcheck" if not sin_hc else f"sin healthcheck: {sin_hc}")
    mal_dep = []
    for n, s in sv.items():
        dep = s.get("depends_on")
        if not dep:
            continue
        if isinstance(dep, list):
            dep = {d: {} for d in dep}
        for d, cond in dep.items():
            if "healthcheck" in sv.get(d, {}) and (cond or {}).get("condition") != "service_healthy":
                mal_dep.append(f"{n}->{d}")
    fila("Orden de arranque (service_healthy)", "Cumple" if not mal_dep else "Falla",
         "dependencias esperan a 'listo'" if not mal_dep else f"esperan solo a 'arrancado': {mal_dep}")
    lit = []
    for n, s in sv.items():
        env = s.get("environment", {})
        env = dict(e.split("=", 1) for e in env if "=" in e) if isinstance(env, list) else env
        for k, v in (env or {}).items():
            if re.search(r"PASSWORD|TOKEN|SECRET|ADMIN_USER", k) and not str(v).startswith("${"):
                lit.append(f"{n}.{k}")
    fila("Secretos fuera del archivo", "Cumple" if not lit else "Falla", "credenciales leidas de variables" if not lit else f"literales en el archivo: {lit}")
    pub = [f"{n}:{p}" for n, s in sv.items() for p in s.get("ports", [])]
    fila("Superficie expuesta (puertos)", "Revisar", f"{len(pub)} puertos publicados; registrar como riesgo aceptado: {pub}")
    tabla_h = pd.DataFrame(filas_h)
    ok_h = int((tabla_h["resultado"] == "Cumple").sum()); tot_h = int((tabla_h["resultado"] != "Revisar").sum())
    print(tabla_h[["punto", "resultado"]].to_string(index=False))
    print(f"\nHardening: {ok_h}/{tot_h} puntos cumplen")
    manifiesto["hardening"] = {"disponible": True, "ok": ok_h, "total": tot_h, "puntos": filas_h}
    tabla_h

                              punto resultado
            Version de las imagenes    Cumple
                           Reinicio    Cumple
                Salud (healthcheck)    Cumple
Orden de arranque (service_healthy)    Cumple
         Secretos fuera del archivo    Cumple
      Superficie expuesta (puertos)   Revisar

Hardening: 5/5 puntos cumplen


## 7. Matriz de trazabilidad, manifiesto y comparación de corridas

In [13]:
b, s_ = manifiesto["batch"], manifiesto["streaming"]
matriz = {
    "variable": "Volumen operado y monto por operacion en BTC/USDT (Binance)",
    "preguntaCentral": "¿Como se comporta el volumen operado y que tan predecible es el monto de una operacion?",
    "integrante": INTEGRANTE, "dimension": DIMENSION, "preguntaDimension": "¿Se puede clasificar probabilísticamente una operación institucional/ballena (> 50 000 USDT) según la liquidez y la dirección del ejecutor?",
    "indicadorU1": {"nombre": "Recall de ballenas (offline)", "valor": b["indicadorU1"], "modelo": b["modeloU1"],
                    "fuenteTecnica": "CSV Binance 2026-01-05 -> Spark batch (Bronze/Silver/Gold) -> modelo MLlib",
                    "evidencia": [RUTA_METRICAS_U1, f"notebook U1_Integrante{INTEGRANTE}"]},
    "indicadorU2": {"nombre": "Recall de ballenas (eventos capturados en vivo)", "valor": b["indicadorU2"],
                    "valorVentana": s_["ventana"].get("recall_pos"),
                    "fuenteTecnica": "WebSocket Binance -> Kafka -> Spark Structured Streaming (ventanas, watermark, checkpoint) -> modelo U1 + serie de tiempo",
                    "evidencia": [RUTA_EV, f"{OUT3}/manifiesto_{RUN_ID}.json", f"{OUT}/resumen_u2.json"]},
    "serieTiempo": {"objetivo": TARGET_COL, "maeModelo": s_["maeTs"], "maePersistencia": s_["maePersistencia"]},
}
def f(v):
    return "n/d" if v is None else (f"{v:.4f}" if isinstance(v, float) else str(v))
md_fila = (f"| {matriz['variable']} | {DIMENSION} (Integrante {INTEGRANTE}) | {matriz['indicadorU1']['nombre']}: {f(b['indicadorU1'])} | "
           f"{matriz['indicadorU2']['nombre']}: {f(b['indicadorU2'])} (ventana: {f(s_['ventana'].get('recall_pos'))}) | "
           f"Batch: Spark sobre CSV; Streaming: Kafka + Spark Structured Streaming | manifiesto_{RUN_ID}.json |")
cab = "| Variable | Dimension | Indicador U1 (batch) | Indicador U2 (streaming) | Fuente y tecnica | Evidencia |\n|---|---|---|---|---|---|\n"
with open(f"{OUT3}/matriz_trazabilidad_integrante{INTEGRANTE}.md", "w", encoding="utf-8") as fh:
    fh.write(f"# Matriz de trazabilidad - Integrante {INTEGRANTE}\n\n**Pregunta central:** {matriz['preguntaCentral']}\n\n"
             f"**Pregunta de la dimension:** {matriz['preguntaDimension']}\n\n" + cab + md_fila + "\n")
json.dump(matriz, open(f"{OUT3}/matriz_trazabilidad_integrante{INTEGRANTE}.json", "w", encoding="utf-8"), indent=2, ensure_ascii=False, default=str)
manifiesto["matrizTrazabilidad"] = matriz
print(cab + md_fila)

| Variable | Dimension | Indicador U1 (batch) | Indicador U2 (streaming) | Fuente y tecnica | Evidencia |
|---|---|---|---|---|---|
| Volumen operado y monto por operacion en BTC/USDT (Binance) | ballenas (Integrante 2) | Recall de ballenas (offline): 0.9940 | Recall de ballenas (eventos capturados en vivo): n/d (ventana: n/d) | Batch: Spark sobre CSV; Streaming: Kafka + Spark Structured Streaming | manifiesto_20261113T195359Z.json |


In [14]:
manifiesto["duracionTotalS"] = round(time.time() - INICIO, 1)
ruta_manifiesto = f"{OUT3}/manifiesto_{RUN_ID}.json"
with open(ruta_manifiesto, "w", encoding="utf-8") as fh:
    json.dump(manifiesto, fh, indent=2, ensure_ascii=False, default=str)
print("Manifiesto escrito en", ruta_manifiesto)
print(json.dumps({k: manifiesto[k] for k in ("runId", "sparkVersion", "duracionTotalS")}, indent=2))

Manifiesto escrito en /home/jovyan/work/artifacts/integrante2/u3/manifiesto_20261113T195359Z.json
{
  "runId": "20261113T195359Z",
  "sparkVersion": "3.5.0",
  "duracionTotalS": 101.0
}


### Comparación con la corrida anterior (prueba de no regresión)
Diferencias **esperables**: valores del stream en vivo y duración. **Regresión**: una compuerta que antes pasaba y ahora no, o un stream que procesa cero filas.

In [15]:
# Prueba de NO REGRESION: compara esta corrida con la anterior. Diferencia esperable != regresion.
manifs = sorted(glob.glob(f"{OUT3}/manifiesto_*.json"))
def get(m, ruta):
    for k in ruta.split("."):
        m = (m or {}).get(k) if isinstance(m, dict) else None
    return m

if len(manifs) < 2:
    print("Solo hay un manifiesto. Vuelve a ejecutar el notebook para comparar dos corridas (piden al menos dos).")
    comparacion = None
else:
    ant, act = (json.load(open(p)) for p in manifs[-2:])
    CAMPOS = [("batch.filas", "batch"), ("batch.latenciaP50Ms", "batch"), ("batch.latenciaP95Ms", "batch"),
              ("batch.indicadorU2", "batch"), ("streaming.filas", "stream"), ("streaming.maeTs", "stream"),
              ("streaming.maePersistencia", "stream"), ("duracionTotalS", "tiempo")]
    filas = []
    for ruta, cat in CAMPOS:
        a, b_ = get(ant, ruta), get(act, ruta)
        if cat == "batch":
            tipo = "sin cambio (esperado: mismo Parquet)" if a == b_ else "cambio: ¿se volvio a ejecutar U2?"
        elif cat == "stream":
            tipo = "REGRESION" if (ruta == "streaming.filas" and not b_) else ("sin cambio" if a == b_ else "esperable (stream en vivo)")
        else:
            tipo = "esperable (carga de la maquina)"
        filas.append({"campo": ruta, "anterior": a, "actual": b_, "lectura": tipo})
    for n in sorted(set(ant["compuertas"]) | set(act["compuertas"])):
        a, b_ = ant["compuertas"].get(n), act["compuertas"].get(n)
        filas.append({"campo": "compuerta: " + n, "anterior": a, "actual": b_,
                      "lectura": "REGRESION" if (a is True and b_ is not True) else ("OK" if b_ else "FALLA")})
    comparacion = pd.DataFrame(filas)
    hay_regresion = bool((comparacion["lectura"] == "REGRESION").any())
    print(f"Comparando {os.path.basename(manifs[-2])}  ->  {os.path.basename(manifs[-1])}")
    print("VEREDICTO:", "HAY REGRESION" if hay_regresion else "SIN REGRESION (las diferencias son las esperables de un stream en vivo)")
    manifiesto["comparacion"] = {"anterior": os.path.basename(manifs[-2]), "veredicto": "regresion" if hay_regresion else "sin regresion"}
comparacion

Comparando manifiesto_20261113T192059Z.json  ->  manifiesto_20261113T195359Z.json
VEREDICTO: SIN REGRESION (las diferencias son las esperables de un stream en vivo)


,campo,anterior,actual,lectura
0,batch.filas,1870,743,cambio: ¿se volvio a ejecutar U2?
1,batch.latenciaP50Ms,3361.0,2673.0,cambio: ¿se volvio a ejecutar U2?
2,batch.latenciaP95Ms,8526.0,5025.0,cambio: ¿se volvio a ejecutar U2?
3,batch.indicadorU2,None,None,sin cambio (esperado: mismo Parquet)
4,streaming.filas,1696,1497,esperable (stream en vivo)
5,streaming.maeTs,0.327116,0.265639,esperable (stream en vivo)
6,streaming.maePersistencia,0.333333,0.0,esperable (stream en vivo)
7,duracionTotalS,101.6,101.0,esperable (carga de la maquina)
8,compuerta: Eventos de streaming de U2 persisti...,True,True,OK
9,compuerta: Grafana saludable,True,True,OK


In [16]:
filas_b = []
for a in manifiesto["advertencias"]:
    filas_b.append((a, "Distribucion o dato que las comprobaciones de nulos y rango no detectan", "Percentiles + advertencia/compuerta de plausibilidad", "Celda de batch: p50, p95 y advertencia"))
for p in manifiesto["hardening"].get("puntos", []):
    if p["resultado"] == "Falla":
        filas_b.append((f"Hardening: {p['punto']} - {p['detalle']}", "Configuracion minima de clase", "Corregir en compose.yml (S14, 3.3-3.4)", "Repetir la ejecucion integrada y comparar manifiestos"))
txt = "| # | Hallazgo | Causa | Correccion | Prueba de correccion |\n|---|---|---|---|---|\n"
for i, f_ in enumerate(filas_b, 1):
    txt += f"| {i} | " + " | ".join(f_) + " |\n"
txt += "| _ | _(completa aqui los hallazgos propios)_ | | | |\n"
with open(f"{OUT3}/bitacora_integrante{INTEGRANTE}.md", "w", encoding="utf-8") as fh:
    fh.write(f"# Bitacora de integracion - Integrante {INTEGRANTE} - {RUN_ID}\n\n" + txt)
print(txt)

| # | Hallazgo | Causa | Correccion | Prueba de correccion |
|---|---|---|---|---|
| 1 | indicador recall_pos no calculable sobre los eventos capturados (clase positiva ausente) | Distribucion o dato que las comprobaciones de nulos y rango no detectan | Percentiles + advertencia/compuerta de plausibilidad | Celda de batch: p50, p95 y advertencia |
| _ | _(completa aqui los hallazgos propios)_ | | | |



### Tablero de integración en Grafana

Importa `dashboard_criptoanalytics_u3.json`. Tiene los paneles de la integración con `instance="integrante2"`: **latencia p50 frente a p95**, ratio de válidos, indicadores U1 y U2, error de la serie de tiempo contra la persistencia, compuertas superadas y la **antigüedad de la última ejecución** (`time() - cripto_int_ultima_ejecucion_timestamp`). Este último panel es el que evita que un valor viejo del Pushgateway parezca vivo.

## 8. Cierre: criterios de aceptación de U3

In [17]:
for q in spark.streams.active:
    q.stop()
M = manifiesto
checks = {
    "Al menos 4 compuertas previas, todas OK": len(M["compuertas"]) >= 4 and all(M["compuertas"].values()),
    "Una compuerta demostrada FALLANDO (detuvo la ejecucion)": M["demostracionFalla"].get("detuvo") is True,
    "KPI de latencia con percentiles (p50/p95) y compuerta de plausibilidad": M["batch"].get("latenciaP95Ms") is not None and any("p50" in k for k in M["batch"]["calidad"]),
    "Batch (U1) + streaming (U2) + modelo ejecutados de punta a punta": M["streaming"].get("filas", 0) > 0 and M["batch"].get("indicadorU1") is not None,
    "Resultados confirmados en Prometheus con el valor recien publicado": M["observabilidad"].get("todasEnPrometheus") is True,
    "Manifiesto escrito": os.path.isfile(f"{OUT3}/manifiesto_{RUN_ID}.json"),
    "Dos corridas comparadas (prueba de no regresion)": "comparacion" in M,
    "Hardening evaluado sobre el compose.yml real": M["hardening"].get("disponible") is True,
    "Matriz de trazabilidad generada": "matrizTrazabilidad" in M,
}
for k, v in checks.items():
    print(("[OK] " if v else "[ ] ") + k)
M["checklistU3"] = checks
json.dump(M, open(f"{OUT3}/manifiesto_{RUN_ID}.json", "w", encoding="utf-8"), indent=2, ensure_ascii=False, default=str)
json.dump({"runId": RUN_ID, "checklist": checks}, open(f"{OUT3}/resumen_u3.json", "w"), indent=2)
print("\nArtefactos de U3 de este integrante:")
for p in sorted(os.listdir(OUT3)):
    print("  ", p)

[OK] Al menos 4 compuertas previas, todas OK
[OK] Una compuerta demostrada FALLANDO (detuvo la ejecucion)
[OK] KPI de latencia con percentiles (p50/p95) y compuerta de plausibilidad
[OK] Batch (U1) + streaming (U2) + modelo ejecutados de punta a punta
[OK] Resultados confirmados en Prometheus con el valor recien publicado
[OK] Manifiesto escrito
[OK] Dos corridas comparadas (prueba de no regresion)
[OK] Hardening evaluado sobre el compose.yml real
[OK] Matriz de trazabilidad generada

Artefactos de U3 de este integrante:
   bitacora_integrante2.md
   checkpoints
   manifiesto_20261113T192059Z.json
   manifiesto_20261113T195359Z.json
   matriz_trazabilidad_integrante2.json
   matriz_trazabilidad_integrante2.md
   resumen_u3.json


### Para la sustentación (S15)
- **Qué componente fue el más difícil de integrar** y por qué (¿el modelo recargado en streaming? ¿las unidades de tiempo? ¿la deriva entre el CSV y el flujo?).
- **Compuerta que detiene frente a la que solo advierte:** el p50 detiene; el p95 advierte. Defiende la diferencia.
- **Por qué el promedio de latencia no representa a los eventos** y qué métrica se usa en su lugar.
- **Prueba de no regresión:** qué diferencias entre las dos corridas eran esperables y cuáles no.
- **Riesgo aceptado** (puertos expuestos, Kafka sin autenticación) con su justificación y qué cambiarías en producción.
- **Fila más débil de la matriz de trazabilidad** y cómo la fortalecerías.